# Qwen3.8-9B Recurrent-Loop (`K=1` vs `K=6`) — Free Colab T4 Demo

Run **Qwen3.8-9B** with **1 to 6 internal reasoning loops** (`152` virtual layers at `K=6` reusing the same `32` quantized blocks) on a free Google Colab **T4 GPU (16 GB VRAM)**.

- **Engine**: [`CympUser/llama.cpp-qwen-loop`](https://github.com/CympUser/llama.cpp-qwen-loop)
- **Base Model**: [`mradermacher/Qwen3.8-9B-Distill-uncensored-heretic-GGUF`](https://huggingface.co/mradermacher/Qwen3.8-9B-Distill-uncensored-heretic-GGUF) (`Q4_K_S`)
- **Loop-6 Adapter (`45.27 MB`)**: [`Vnswz/Qwen3.8-9B-Loop6-Adapter`](https://huggingface.co/Vnswz/Qwen3.8-9B-Loop6-Adapter)

> Make sure your Colab runtime is set to **Runtime → Change runtime type → T4 GPU**.

In [ ]:
# 1. Check GPU & build llama-cli with CUDA (T4 = sm_75)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!git clone --depth 1 --branch qwen35-loop https://github.com/CympUser/llama.cpp-qwen-loop.git
%cd llama.cpp-qwen-loop
!cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 -DLLAMA_CURL=OFF
!cmake --build build --config Release --target llama-cli -j$(nproc)

In [ ]:
# 2. Download Base GGUF (5.23 GB) and Loop-6 Adapter (45 MB)
!mkdir -p models
!wget -q --show-progress -O models/Qwen3.8-9B-Distill-uncensored-heretic.Q4_K_S.gguf \
  https://huggingface.co/mradermacher/Qwen3.8-9B-Distill-uncensored-heretic-GGUF/resolve/main/Qwen3.8-9B-Distill-uncensored-heretic.Q4_K_S.gguf
!wget -q --show-progress -O models/qwen3.8-9b-heretic-loop6-lora.gguf \
  https://huggingface.co/Vnswz/Qwen3.8-9B-Loop6-Adapter/resolve/main/qwen3.8-9b-heretic-loop6-lora.gguf
!ls -lh models/

In [ ]:
# 3. Helper to run the exact same prompt at any loop depth K in 1..6
import subprocess

def run_qwen_loop(prompt: str, num_loops: int = 6, max_tokens: int = 512) -> str:
    cmd = [
        "./build/bin/llama-cli",
        "-m", "models/Qwen3.8-9B-Distill-uncensored-heretic.Q4_K_S.gguf",
        "--lora", "models/qwen3.8-9b-heretic-loop6-lora.gguf",
        "--override-kv", f"qwen35.num_loops=int:{num_loops},qwen35.recurrent_kv_cache_mode=str:per_loop",
        "-c", "8192", "-ngl", "999", "--fit", "off",
        "--cache-type-k", "q8_0", "--cache-type-v", "q8_0",
        "--temp", "0.0", "-n", str(max_tokens),
        "-st", "--simple-io", "-no-cnv",
        "-p", f"<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n"
    ]
    res = subprocess.run(cmd, capture_output=True, text=True)
    return res.stdout

In [ ]:
# 4. Side-by-side comparison: 4-Item Deductive Ordering (K=1 vs K=6)
logic_prompt = (
    "Four servers (Alpha, Beta, Gamma, Delta) are ranked from fastest to slowest. "
    "Clues: (1) Beta is faster than Gamma. (2) Alpha is slower than Gamma. "
    "(3) Delta is faster than Beta. "
    "Reply ONLY with the final order from fastest to slowest using '>' (e.g., A>B>C>D)."
)

print("=== K=1 (Base single-pass) ===")
print(run_qwen_loop(logic_prompt, num_loops=1, max_tokens=384))

print("=== K=6 (6 Internal Reasoning Loops) ===")
print(run_qwen_loop(logic_prompt, num_loops=6, max_tokens=384))

In [ ]:
# 5. Try your own prompt! Change NUM_LOOPS between 1 and 6
MY_PROMPT = "Explain in 3 concise bullet points why recurrent-depth transformers save VRAM compared to deeper models."
NUM_LOOPS = 6  # Try 1, 2, 4, or 6

print(run_qwen_loop(MY_PROMPT, num_loops=NUM_LOOPS, max_tokens=512))